<div dir="rtl" align="right">

# مُرشِّحُ النطاقِ \(Band-Pass Filter\)

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 7

---

## نظرةٌ عامّةٌ

يَجمعُ مُرشِّحُ النطاقِ بين مُرشِّحٍ عالٍ وآخَرَ منخفضٍ في عمليةٍ واحدةٍ. يُبقي التردداتِ بين 1 و40 Hz فقط، وهي تُغطّي الموجاتِ الدماغيةَ الرئيسةَ:

| النطاقُ | الترددُ | يرتبطُ بـ |
| ------- | ------- | --------- |
| دلتا | 1 إلى 4 Hz | النومُ العميقُ |
| ثيتا | 4 إلى 8 Hz | النعاسُ، الذاكرةُ |
| ألفا | 8 إلى 13 Hz | الاسترخاءُ (إغلاقُ العينينِ) |
| بيتا | 13 إلى 30 Hz | التفكيرُ النشطُ، التركيزُ |

## المُخرجاتُ المُتوقّعةُ

تكونُ الإشارةُ المُرشَّحةُ:
- مُتمركزةً حولَ الصفرِ (لا إزاحةَ تيارٍ، كالمُرشِّحِ العاليِّ)
- ناعمةً (لا ضجيجَ عاليَ الترددِ، كالمُرشِّحِ المنخفضِ)
- نظيفةً بما يكفي لظهورِ الشوائبِ مثلَ طَرْفَاتِ العينِ والمضغِ، التي كانت مَخفيةً بالضجيجِ قبلَ ذلك

## لماذا لا نَكتفي بالمُرشِّحَين منفصلينِ؟

يمكنُ ذلك والنتيجةُ واحدةٌ، لكنَّ مُرشِّحَ النطاقِ يُؤدّي العملَ في استدعاءِ دالةٍ واحدٍ وهو أكثرُ كفاءةً. نُظهرُ الطريقتينِ لتفهمَ ما يحدثُ في الخلفيةِ.

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly wfdb

<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 7

<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2، قناةَ P4 (المنطقةُ الجداريةُ).

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=7, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')

<div dir="rtl" align="right">

## 4. تطبيقُ مُرشِّحِ النطاقِ (1 إلى 40 Hz)

نَستخدمُ `signal.butter` مع `btype='band'` ونُمرِّرُ تردّديِ القطعِ كقائمةٍ. تضمنُ دالةُ `filtfilt` عدمَ وجودِ تأخيرٍ في الطورِ.

</div>

In [ ]:
from scipy import signal

def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs  # Nyquist = 100 Hz
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band', analog=False)
    return signal.filtfilt(b, a, data)

filtered_bp = butter_bandpass_filter(
    channel_data, lowcut=1.0, highcut=40.0, fs=fs
)
print(f'Band-pass filter applied: {1.0} to {40.0} Hz, order {4}')
print(f'Nyquist frequency: {0.5*fs} Hz')

<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ: الإشارةُ الخامُ مقابل المُرشَّحةُ

**علامَ تُلاحظُ؟**
- الإشارةُ الخامُ (أعلى) فيها انجرافٌ وضجيجٌ معًا
- الإشارةُ المُرشَّحةُ (أسفل) مُتمركزةٌ حولَ الصفرِ وناعمةٌ
- قد تَظهرُ شوائبُ (ارتفاعاتٌ مُفاجئةٌ) في الإشارةِ المُرشَّحةِ كانت مَخفيةً بالضجيجِ قبلَ ذلك. قد تكونُ طَرْفَاتِ عينٍ أو حركاتِ عضلاتٍ. سنتعاملُ مع الشوائبِ في فصلٍ لاحقٍ.

استخدمْ أداةَ التكبيرِ لفحصِ نطاقاتَ زمنيةٍ مُحدّدةٍ، ومرّرْ المؤشّرَ فوقَ الإشارةِ لرؤيةِ القيمِ الدقيقةِ.

</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = timestamps[:n_plot] / 1000.0

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Raw EEG (P4)', 'Band-pass filtered (1 to 40 Hz)'))

fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot],
                         name='Raw', line=dict(color='gray', width=0.5)),
               row=1, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=filtered_bp[:n_plot],
                         name='Band-pass', line=dict(color='blue', width=0.5)),
               row=2, col=1)

fig.update_layout(height=600, title_text='Band-Pass Filter: Raw vs Filtered',
                  xaxis2_title='Time (s)', yaxis_title='EEG (uV)',
                  yaxis2_title='EEG (uV)')
fig.show()

<div dir="rtl" align="right">

## 6. مقارنةُ المُرشِّحاتِ الثلاثةِ

نُرسمُ الإشارةَ الخامَ ونتائجَ المُرشِّحاتِ الثلاثةِ معًا لرؤيةِ التطوّرِ:
1. **خامٌ** (رماديٌّ): انجرافٌ وضجيجٌ
2. **عالٍ فقط** (أخضرُ): اختفى الانجرافُ وبقيَ الضجيجُ
3. **عالٍ ثمَّ منخفضٌ** (أحمرُ): اختفى الانجرافُ والضجيجُ
4. **نطاقٌ** (أزرقُ): نفسُ النتيجةِ لكن في خطوةٍ واحدةٍ

لاحظْ كيفَ يَتطابقُ الخطّانِ الأحمرُ والأزرقُ تقريبًا. هذا يُؤكّدُ أنَّ مُرشِّحَ النطاقِ يُعادلُ تطبيقَ المُرشِّحَين معًا.

</div>

In [ ]:
# Apply all three filters for comparison
def butter_highpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    b, a = signal.butter(order, cutoff / nyq, btype='high', analog=False)
    return signal.filtfilt(b, a, data)

def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    b, a = signal.butter(order, cutoff / nyq, btype='low', analog=False)
    return signal.filtfilt(b, a, data)

hp = butter_highpass_filter(channel_data, 1.0, fs)
hp_lp = butter_lowpass_filter(hp, 40.0, fs)
bp = butter_bandpass_filter(channel_data, 1.0, 40.0, fs)

n_plot = min(3000, len(channel_data))
t_sec = timestamps[:n_plot] / 1000.0

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot], name='Raw',
                         line=dict(color='gray', width=0.5), opacity=0.5))
fig.add_trace(go.Scatter(x=t_sec, y=hp[:n_plot], name='High-pass only',
                         line=dict(color='green', width=0.5)))
fig.add_trace(go.Scatter(x=t_sec, y=hp_lp[:n_plot], name='HP + LP',
                         line=dict(color='red', width=0.5)))
fig.add_trace(go.Scatter(x=t_sec, y=bp[:n_plot], name='Band-pass',
                         line=dict(color='blue', width=0.5)))

fig.update_layout(height=500, title_text='All filters compared (zoom in to see differences)',
                  xaxis_title='Time (s)', yaxis_title='EEG (uV)')
fig.show()

<div dir="rtl" align="right">

## 7. خلاصةٌ

- يَجمعُ مُرشِّحُ النطاقِ (1 إلى 40 Hz) بين المُرشِّحِ العاليِّ والمنخفضِ في خطوةٍ واحدةٍ
- النتيجةُ مُطابقةٌ لتطبيقِهما بالتتابعِ
- الإشارةُ المُرشَّحةُ نظيفةٌ بما يكفي لظهورِ الشوائبِ (طَرْفَاتِ العينِ، حركاتِ العضلاتِ)
- هذه أولى خطواتِ المعالجةِ المُسبقةِ في أيِّ خطِّ معالجةِ EEG
- يُغطّي الفصلُ التالي مُرشِّحاتِ التنعيمِ (المتوسطُ المتحرّكُ، Gaussian) التي تُقلّلُ الضجيجَ أكثرَ

</div>